# 2.5 視窗多長，才把必要線索交給模型？


本節用一個小資料世界：紅色物體後接「圓」，藍色物體後接「方」。材料與答案是「紅色物體是→圓」和「藍色物體是→方」；這是例子的答案規則，不是說現實中顏色決定形狀。

如果只給最後一字「是」，模型收到完全相同的輸入，卻希望它分別答圓與方。再練多久也不能知道這一回是哪種顏色。因此先問：關鍵的紅或藍，有沒有送進去？

預測時能使用的前文叫**上下文**；實際取出最近幾個位置的範圍叫**視窗**。下面只整理材料，沒有訓練。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
examples = [("紅色物體是", "圓"), ("藍色物體是", "方")]
for length in [1, 3, 5]:
    visible = []
    for prefix, answer in examples:
        visible.append((prefix[-length:], answer))
    print(length, visible)

一字視窗的兩筆都是「是」，三字都是「物體是」；五字才分別得到「紅色物體是」與「藍色物體是」。四字也不夠，因為都剩「色物體是」。


In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/rewrite-02-visible-window.svg")))

圖中灰字沒有送入，藍框是實際輸入。相同輸入只能得到相同候選分配；模型能給圓、方各一些機率，卻無法根據不存在的顏色線索穩定挑對。五字則使兩題可區分，提供學會的機會，還不等於已經用好線索。

多看字也有成本。每字 D 格、視窗 C 字，拼接就有 C×D 格；若接 H 個輸出，每套配方要為每個輸入存乘數，因此需要 H×C×D 個權重。其他固定時，視窗翻倍會讓這部分權重翻倍。

現有另一個小實驗讓模型看 12 篇欄位短文，如「顏色=紅；形狀=圓。」。三個模型分別看最近 1、3、5 字，都使用相同 9 篇訓練、1 篇驗證，更新 200 次。它們先查表、拼接，再經線性與非線性層猜下一字；這樣的小網路叫**多層感知器**，簡稱 MLP。

| 可見視窗 | 參數格數 | 9 篇訓練平均代價 | 1 篇驗證平均代價 |
| --- | ---: | ---: | ---: |
| 1 字 | 833 | 0.33352 | 0.43103 |
| 3 字 | 1,345 | 0.21302 | 0.30577 |
| 5 字 | 1,857 | 0.19883 | 0.51252 |

從 1 字到 3 字，兩份代價都降低；到 5 字，訓練繼續改善，驗證卻升高。更多線索沒有保證新材料更好。而視窗變長時參數也增加，驗證只有一篇，這份結果不能證明所有任務三字最好。

練習把前文改成「紅色的小小物體是」與「藍色的小小物體是」，答案不變。整句八字，紅與藍在第一格；最近七字仍相同，八字才可區分。先手數再新增視窗 7、8 核對，把資訊是否可見與學習是否成功分開。

<details>
<summary>補充與重做</summary>

短文實驗使用特徵寬度 16、種子 42，完整切分、重做命令與另兩篇最後檢查見 [T.3](https://birdhackor.github.io/tiny-perceptron-vlm/training.html#T.3)。原來的曲線圖保留同一表格資料：[視窗訓練與驗證代價](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/course/figures/window_training.svg)，橫軸是視窗，不是訓練時間。這組比較同時改了視窗與參數量；要回答嚴格的單因素因果問題，還需別的受控設計。

</details>
